In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# DEPENDENCIES

In [3]:
from bs4 import BeautifulSoup
import requests
from fake_useragent import UserAgent
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter
import pandas as pd
import numpy as np
import threading 
from concurrent.futures import ThreadPoolExecutor
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [4]:
lock = threading.Lock()

fight_details = []
new_fight_links_all = []
winner_names = []
fighter_detail_data = []

In [5]:
MAX_THREADS = 10 # change this to adjust the number of concurrent threads

ua = UserAgent()
chrome = ua.chrome

HEADER = {
    'User-Agent' : chrome
}

In [6]:
def create_session(): # Create a session with retry strategy
    """Create a requests session with retry strategy for handling network issues."""
    # This function sets up a session with a retry strategy to handle network issues. 
    
    session = requests.Session()
    retry_strat = Retry(
        backoff_factor=1,
        total=3,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=['GET']
    )
    adapter = HTTPAdapter(max_retries= retry_strat)
    session.mount('https://', adapter)
    session.mount('http://', adapter)
    return session

session = create_session()

# Scraping the event links

In [7]:
# OLD LOGIC - no longer works

# ufc_link = "http://ufcstats.com/statistics/events/completed?page=all"

# respone = session.get(ufc_link)

# text = respone.text
# soup = BeautifulSoup(text, 'lxml')

# event_links_soup = soup.find_all('a', class_ = 'b-link b-link_style_black')

# event_links = [link['href'] for link in event_links_soup] # Extracting href attributes from the links

# print(len(event_links), "events found")

In [8]:
import subprocess
import sys
from pathlib import Path

backend_dir = Path.cwd()

script_path = backend_dir / "get_event_links.py"
links_path = backend_dir / "data" / "event_links.txt"

print("Getting UFCStats event links...")

result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("get_event_links.py failed")

with open(links_path, "r") as f:
    event_links = [line.strip() for line in f if line.strip()]

print(f"{len(event_links)} events loaded.")

Getting UFCStats event links...
786 events found
http://ufcstats.com/event-details/9d61d8cb1c354867
http://ufcstats.com/event-details/a0a69dc9914ef6e1
http://ufcstats.com/event-details/b96619b3acd7d9da
Saved event links to data\event_links.txt

786 events loaded.


# Scraping the event info

In [9]:
# def get_event_data(item): # Function to scrape event data
#     """Scrape event data from the given link."""
#     idx, link = item
#     link = link.strip()
#     response = session.get(link, headers=HEADER, timeout= 15)
#     response.raise_for_status()
#     if (response.status_code == 200):
#         soup = BeautifulSoup(response.text, 'lxml')
                
#         event_id = link[-16:]
#         date_loc_list = soup.find_all('li', 'b-list__box-list-item')
#         date = date_loc_list[0].text.replace("Date:", "").strip()
#         location = date_loc_list[1].text.replace("Location:", "").strip()
#         fight_links = soup.find_all('tr', class_ = 'b-fight-details__table-row b-fight-details__table-row__hover js-fight-details-click')
#         for i in fight_links:
#             winner_name = None
#             winner_id = None
#             w_l_d = i.find('i', class_ = "b-flag__text").text
#             fight_id = i['data-link'][-16:]
#             # print(w_l_d)
#             if w_l_d == "win":
#                 players = i.find('td', class_ = "b-fight-details__table-col l-page_align_left")
#                 players = players.find_all('a', class_= "b-link b-link_style_black")
#                 winner_name = players[0].text.strip()
#                 winner_id = players[0]['href'][-16:]
#             # Making the data
#             data_dic = {
#                 "event_id" : event_id,
#                 "fight_id" : fight_id,
#                 "date" : date,
#                 "location" : location,
#                 "winner" : winner_name,
#                 "winner_id" : winner_id
#             }
#             new_fight_links_all.append(i['data-link'])
#             winner_names.append(data_dic)
#         # print(f"Scrapped : {link}, {idx+1} / {len(event_links)}")
#         idx += 1
#     else:
#         print("Could'nt retrive the link." + str(response.status_code))

# with ThreadPoolExecutor(max_workers= MAX_THREADS) as executor:
#     results = [executor.submit(get_event_data, item) for item in enumerate(event_links)]
#     for r in results:
#         r.result()

# df_winner = pd.DataFrame(data=winner_names)
# df_winner.to_csv("event_details.csv", index = False)
# print(f"Successfully scrapped {len(df_winner)} event data.")
# df_winner

In [ ]:
import subprocess
import sys
from pathlib import Path
import pandas as pd

backend_dir = Path.cwd()

script_path = backend_dir / "scrape_events.py"
event_details_path = backend_dir / "data" / "event_details.csv"
fight_links_path = backend_dir / "data" / "fight_links.txt"

print("Scraping UFCStats event data...")

result = subprocess.run(
    [sys.executable, str(script_path)],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("scrape_events.py failed")

df_winner = pd.read_csv(event_details_path)

with open(fight_links_path, "r") as f:
    new_fight_links_all = [
        line.strip()
        for line in f
        if line.strip()
    ]

df_event = pd.read_csv(backend_dir / "data" / "event_details.csv")
print(f"Event/fight records loaded: {len(df_winner)}")
print(f"Fight links loaded: {len(new_fight_links_all)}")

Scraping UFCStats event data...
Loaded 786 event links
Found 8675 existing fight records across previously-scraped events
Found 8832 existing fight links
15 events remaining to scrape

Scraping 15 events with 6 concurrent browser workers...
FAILED [1] - Title: Loadingâ€¦ HTML: 2994
FAILED [6] http://ufcstats.com/event-details/f354c50b8d63d9b3
Error: Page.goto: net::ERR_ABORTED at http://ufcstats.com/event-details/f354c50b8d63d9b3
Call log:
  - navigating to "http://ufcstats.com/event-details/f354c50b8d63d9b3", waiting until "domcontentloaded"

FAILED [0] - Title: Loadingâ€¦ HTML: 2994
FAILED [7] - Title: None HTML: 66
FAILED [11] - Title: None HTML: 66
FAILED [5] http://ufcstats.com/event-details/681d07e328798ec0
Error: Page.content: Unable to retrieve content because the page is navigating and changing the content.
FAILED [13] - Title: None HTML: 66
FAILED [14] - Title: None HTML: 66
FINAL saved: 8758 fight records, 8832 fight links

--------------------------------
SCRAPING COMPLETE


# Scraping the fight info

In [11]:
# def get_fight_data(item): # Function to scrape fight data
#     """Scrape fight data from the given link."""
#     idx, link = item
#     link = link.strip()
#     try:
#         response = session.get(link, headers=HEADER, timeout=15)
#         response.raise_for_status() 
        
#         soup = BeautifulSoup(response.text, 'lxml')
        
#         # event name
#         event_name = soup.find('a', class_ = "b-link").text.strip()
#         # event id
#         event_id = soup.find('a', class_ = "b-link")['href'][-16:]
#         # fight id
#         fight_id = link[-16:]
        
#         # fighter names
#         fighter_nams = soup.find_all('a', class_ = 'b-link b-fight-details__person-link')
#         r_name = fighter_nams[0].text.strip()
#         b_name = fighter_nams[1].text.strip()
        
#         # fighter ids
#         r_id = fighter_nams[0]['href'].strip()[-16:]
#         b_id = fighter_nams[1]['href'].strip()[-16:]
        
#         # title fight & division
#         division_info = soup.find('i', class_= 'b-fight-details__fight-title').text.lower()
#         is_title_fight = 0
#         if 'title' in division_info:
#             is_title_fight = 1
#         division_info = division_info.replace('ufc', "")
#         division_info = division_info.replace("title", "")
#         division_info = division_info.replace("bout", "").strip()
        
#         # method
#         method = soup.find('i', style = 'font-style: normal').text.strip()
        
        
#         p_tag_with_fight_detail = soup.find('p', class_ = "b-fight-details__text")
#         fight_details_list = p_tag_with_fight_detail.find_all('i', class_ = 'b-fight-details__text-item')
#         # finish-round
#         finish_round = int(fight_details_list[0].text.lower().replace("round:", "").strip())
#         # match-time
#         match_timestamp = fight_details_list[1].text.lower().replace("time:", "").strip()
#         match_timestamp_splited = match_timestamp.split(":")
#         match_time_sec = int(match_timestamp_splited[0]) * 60 + int(match_timestamp_splited[-1])
#         # total-round
#         total_rounds = fight_details_list[2].text.lower().replace("time format:", "").strip()
#         if total_rounds == "No Time Limit".lower():
#             total_rounds = None
#         else :
#             total_rounds = int(total_rounds[0])
#         # referee
#         referee = fight_details_list[3].text.replace("Referee:", "").strip()
        
        
#         # totals, SIG. STR.
#         tables = soup.find_all('table', style = "width: 745px")
        
#         # TOTALS TABLE
#         if len(tables) > 0:
#             table1 = tables[0]
#             td_1_list = table1.find_all('td', class_ = 'b-fight-details__table-col')
#             # KD
#             kd_players = td_1_list[1].text.split()
#             r_kd, b_kd = int(kd_players[0]), int(kd_players[1])
#             # sig. str.
#             sig_str_players = td_1_list[2].text.split() 
#             r_sig_str_landed = int(sig_str_players[0])
#             r_sig_str_atmpted = int(sig_str_players[2])
#             b_sig_str_landed = int(sig_str_players[3])
#             b_sig_str_atmpted = int(sig_str_players[5])
#             # sig_str_acc
#             sig_str_acc = td_1_list[3].text.split() 
#             r_sig_str_acc = int(sig_str_acc[0].replace("%", "")) if sig_str_acc[0] != "---" else None
#             b_sig_str_acc = int(sig_str_acc[1].replace("%", "")) if sig_str_acc[1] != "---" else None
#             # total-str
#             total_str = td_1_list[4].text.split() 
#             r_total_str_landed = int(total_str[0])
#             r_total_str_atmpted = int(total_str[2])
#             b_total_str_landed = int(total_str[3])
#             b_total_str_atmpted = int(total_str[5])
#             # total-str-acc
#             r_total_str_acc, b_total_str_acc = None, None
#             try:
#                 r_total_str_acc = int(round(r_total_str_landed / r_total_str_atmpted, 2) * 100)
#             except:
#                 pass
#             try:
#                 b_total_str_acc = int(round(b_total_str_landed / b_total_str_atmpted, 2) * 100)
#             except:
#                 pass
#             # TD
#             td_players = td_1_list[5].text.split() 
#             r_td_landed = int(td_players[0])
#             r_td_atmpted = int(td_players[2])
#             b_td_landed = int(td_players[3])
#             b_td_atmpted = int(td_players[5])
#             # td_acc
#             td_acc = td_1_list[6].text.split() 
#             r_td_acc = int(td_acc[0].replace("%", "")) if td_acc[0] != "---" else None
#             b_td_acc = int(td_acc[1].replace("%", "")) if td_acc[1] != "---" else None
#             # sub. att
#             sub_att = td_1_list[7].text.split()
#             r_sub_att, b_sub_att = int(sub_att[0]), int(sub_att[1])
#             # rev
#             rev = td_1_list[8].text.split()
#             r_rev, b_rev = int(rev[0]), int(rev[1])
#             # Ctrl
#             ctrl = td_1_list[9].text.split()
#             r_ctrl = ctrl[0].split(":")
#             r_ctrl = int(r_ctrl[0]) * 60 + int(r_ctrl[1]) if r_ctrl[0] != '--' else None
#             b_ctrl = ctrl[1].split(":")
#             b_ctrl = int(b_ctrl[0]) * 60 + int(b_ctrl[1]) if b_ctrl[0] != '--' else None
            
#             # SIG. STR. TABLE
#             table2 = tables[1]
#             td_2_list = table2.find_all('td', class_ = 'b-fight-details__table-col')
            
#             # HEAD
#             head_list = td_2_list[3].text.split() 
#             r_head_landed = int(head_list[0])
#             r_head_atmpted = int(head_list[2])
#             b_head_landed = int(head_list[3])
#             b_head_atmpted = int(head_list[5])
#             # HEAD
#             r_head_acc, b_head_acc = None, None
#             try:
#                 r_head_acc = int(round(r_head_landed / r_head_atmpted, 2) * 100)
#             except:
#                 pass
#             try:
#                 b_head_acc = int(round(b_head_landed / b_head_atmpted, 2) * 100)
#             except:
#                 pass
            
#             # BODY
#             body_list = td_2_list[4].text.split() 
#             r_body_landed = int(body_list[0])
#             r_body_atmpted = int(body_list[2])
#             b_body_landed = int(body_list[3])
#             b_body_atmpted = int(body_list[5])
#             # BODY ACC
#             r_body_acc, b_body_acc = None, None
#             try:
#                 r_body_acc = int(round(r_body_landed / r_body_atmpted, 2) * 100)
#             except:
#                 pass
#             try:
#                 b_body_acc = int(round(b_body_landed / b_body_atmpted, 2) * 100)
#             except:
#                 pass
            
#             # LEG
#             leg_list = td_2_list[5].text.split() 
#             r_leg_landed = int(leg_list[0])
#             r_leg_atmpted = int(leg_list[2])
#             b_leg_landed = int(leg_list[3])
#             b_leg_atmpted = int(leg_list[5])
#             # LEG ACC
#             r_leg_acc, b_leg_acc = None, None
#             try:
#                 r_leg_acc = int(round(r_leg_landed / r_leg_atmpted, 2) * 100)
#             except:
#                 pass
#             try:
#                 b_leg_acc = int(round(b_leg_landed / b_leg_atmpted, 2) * 100)
#             except:
#                 pass
            
#             # DISTANCE
#             dist_list = td_2_list[6].text.split() 
#             r_dist_landed = int(dist_list[0])
#             r_dist_atmpted = int(dist_list[2])
#             b_dist_landed = int(dist_list[3])
#             b_dist_atmpted = int(dist_list[5])
#             # DIST ACC
#             r_dist_acc, b_dist_acc = None, None
#             try:
#                 r_dist_acc = int(round(r_dist_landed / r_dist_atmpted, 2) * 100)
#             except:
#                 pass
#             try:
#                 b_dist_acc = int(round(b_dist_landed / b_dist_atmpted, 2) * 100)
#             except:
#                 pass
            
#             # CLINCH
#             clinch_list = td_2_list[7].text.split() 
#             r_clinch_landed = int(clinch_list[0])
#             r_clinch_atmpted = int(clinch_list[2])
#             b_clinch_landed = int(clinch_list[3])
#             b_clinch_atmpted = int(clinch_list[5])
#             # CLINCH ACC
#             r_clinch_acc, b_clinch_acc = None, None
#             try:
#                 r_clinch_acc = int(round(r_clinch_landed / r_clinch_atmpted, 2) * 100)
#             except:
#                 pass
#             try:
#                 b_clinch_acc = int(round(b_clinch_landed / b_clinch_atmpted, 2) * 100)
#             except:
#                 pass
            
#             # Ground
#             ground_list = td_2_list[8].text.split() 
#             r_ground_landed = int(ground_list[0])
#             r_ground_atmpted = int(ground_list[2])
#             b_ground_landed = int(ground_list[3])
#             b_ground_atmpted = int(ground_list[5])
#             # Ground ACC
#             r_ground_acc, b_ground_acc = None, None
#             try:
#                 r_ground_acc = int(round(r_ground_landed / r_ground_atmpted, 2) * 100)
#             except:
#                 pass
#             try:
#                 b_ground_acc = int(round(b_ground_landed / b_ground_atmpted, 2) * 100)
#             except:
#                 pass
#         else:
#             r_kd,b_kd = None, None
#             r_sig_str_landed,b_sig_str_landed = None, None
#             r_sig_str_atmpted,b_sig_str_atmpted = None, None
#             r_sig_str_acc,b_sig_str_acc = None, None
#             r_total_str_landed,b_total_str_landed = None, None
#             r_total_str_atmpted,b_total_str_atmpted = None, None
#             r_total_str_acc,b_total_str_acc = None, None
#             r_td_landed,b_td_landed= None, None
#             r_td_atmpted,b_td_atmpted = None, None
#             r_td_acc,b_td_acc= None, None
#             r_sub_att,b_sub_att= None, None
#             r_ctrl,b_ctrl= None, None
            
#             r_head_landed , b_head_landed = None, None
#             r_head_atmpted , b_head_atmpted = None, None
#             r_head_acc , b_head_acc = None, None
#             r_body_landed , b_body_landed = None, None
#             r_body_atmpted , b_body_atmpted = None, None
#             r_body_acc , b_body_acc = None, None
#             r_leg_landed , b_leg_landed = None, None
#             r_leg_atmpted , b_leg_atmpted = None, None
#             r_leg_acc , b_leg_acc = None, None
#             r_dist_landed , b_dist_landed = None, None
#             r_dist_atmpted , b_dist_atmpted = None, None
#             r_dist_acc , b_dist_acc = None, None
#             r_clinch_landed , b_clinch_landed = None, None
#             r_clinch_atmpted , b_clinch_atmpted= None, None
#             r_clinch_acc , b_clinch_acc = None, None
#             r_ground_landed , b_ground_landed = None, None
#             r_ground_atmpted , b_ground_atmpted = None, None
#             r_ground_acc , b_ground_acc = None, None
#             r_landed_head_per , b_landed_head_per = None, None
#             r_landed_body_per , b_landed_body_per= None, None
#             r_landed_leg_per , b_landed_leg_per = None, None
#             r_landed_dist_per , b_landed_dist_per = None, None
#             r_landed_clinch_per , b_landed_clinch_per = None, None
#             r_landed_ground_per , b_landed_ground_per = None, None
        
#         # LANDED-head&dist
#         try:
#             r_landed_head_and_dist_list = soup.find_all('i', class_= "b-fight-details__charts-num b-fight-details__charts-num_style_red b-fight-details__charts-num_pos_left js-red")
#             r_landed_head_per = int(r_landed_head_and_dist_list[0].text.strip().replace("%", ""))
#             r_landed_dist_per = int(r_landed_head_and_dist_list[1].text.strip().replace("%", ""))
#             b_landed_head_and_dist_list = soup.find_all('i', class_= "b-fight-details__charts-num b-fight-details__charts-num_style_blue b-fight-details__charts-num_pos_right js-blue")
#             b_landed_head_per = int(b_landed_head_and_dist_list[0].text.strip().replace("%", ""))
#             b_landed_dist_per = int(b_landed_head_and_dist_list[1].text.strip().replace("%", ""))
#         except:
#             r_landed_head_per, r_landed_dist_per = None, None
#             b_landed_head_per, b_landed_dist_per = None, None
#         # LANDED-Body&Clinch
#         try:
#             r_landed_body_and_clinch_list = soup.find_all('i', class_= "b-fight-details__charts-num b-fight-details__charts-num_style_dark-red b-fight-details__charts-num_pos_left js-red")
#             r_landed_body_per = int(r_landed_body_and_clinch_list[0].text.strip().replace("%", ""))
#             r_landed_clinch_per = int(r_landed_body_and_clinch_list[1].text.strip().replace("%", ""))
#             b_landed_body_and_clinch_list = soup.find_all('i', class_= "b-fight-details__charts-num b-fight-details__charts-num_style_dark-blue b-fight-details__charts-num_pos_right js-blue")
#             b_landed_body_per = int(b_landed_body_and_clinch_list[0].text.strip().replace("%", ""))
#             b_landed_clinch_per = int(b_landed_body_and_clinch_list[1].text.strip().replace("%", ""))
#         except:
#             r_landed_body_per, r_landed_clinch_per = None, None
#             b_landed_body_per, b_landed_clinch_per = None, None
            
#         # LANDED-leg&ground
#         try:
#             r_landed_leg_and_ground_list = soup.find_all('i', class_= "b-fight-details__charts-num b-fight-details__charts-num_style_light-red b-fight-details__charts-num_pos_left js-red")
#             r_landed_leg_per = int(r_landed_leg_and_ground_list[0].text.strip().replace("%", ""))
#             r_landed_ground_per = int(r_landed_leg_and_ground_list[1].text.strip().replace("%", ""))
#             b_landed_leg_and_ground_list = soup.find_all('i', class_= "b-fight-details__charts-num b-fight-details__charts-num_style_light-blue b-fight-details__charts-num_pos_right js-blue")
#             b_landed_leg_per = int(b_landed_leg_and_ground_list[0].text.strip().replace("%", ""))
#             b_landed_ground_per = int(b_landed_leg_and_ground_list[1].text.strip().replace("%", ""))
#         except:
#             # pass
#             r_landed_leg_per, r_landed_ground_per = None, None
#             b_landed_leg_per, b_landed_ground_per = None, None
            
#         # MAKING THE DATA
#         data_dic = {
#             "event_name" : event_name,
#             "event_id" : event_id,
#             "fight_id" : fight_id,
#             "r_name" : r_name,
#             "r_id" : r_id,
#             "b_name" : b_name,
#             "b_id" : b_id,
#             "division" : division_info,
#             "title_fight" : is_title_fight,
#             "method" : method,
#             "finish_round" : finish_round,
#             "match_time_sec" : match_time_sec,
#             "total_rounds" : total_rounds,
#             "referee" : referee,
#             "r_kd" : r_kd,
#             "r_sig_str_landed" : r_sig_str_landed,
#             "r_sig_str_atmpted" : r_sig_str_atmpted,
#             "r_sig_str_acc" : r_sig_str_acc,
#             "r_total_str_landed" : r_total_str_landed,
#             "r_total_str_atmpted" : r_total_str_atmpted,
#             "r_total_str_acc" : r_total_str_acc,
#             "r_td_landed" : r_td_landed,
#             "r_td_atmpted" : r_td_atmpted,
#             "r_td_acc" : r_td_acc,
#             "r_sub_att" : r_sub_att,
#             "r_ctrl" : r_ctrl,
#             "r_head_landed" : r_head_landed,
#             "r_head_atmpted" : r_head_atmpted,
#             "r_head_acc" : r_head_acc,
#             "r_body_landed" : r_body_landed,
#             "r_body_atmpted" : r_body_atmpted,
#             "r_body_acc" : r_body_acc,
#             "r_leg_landed" : r_leg_landed,
#             "r_leg_atmpted" : r_leg_atmpted,
#             "r_leg_acc" : r_leg_acc,
#             "r_dist_landed" : r_dist_landed,
#             "r_dist_atmpted" : r_dist_atmpted,
#             "r_dist_acc" : r_dist_acc,
#             "r_clinch_landed" : r_clinch_landed,
#             "r_clinch_atmpted" : r_clinch_atmpted,
#             "r_clinch_acc" : r_clinch_acc,
#             "r_ground_landed" : r_ground_landed,
#             "r_ground_atmpted" : r_ground_atmpted,
#             "r_ground_acc" : r_ground_acc,
#             "r_landed_head_per" : r_landed_head_per,
#             "r_landed_body_per" : r_landed_body_per,
#             "r_landed_leg_per" : r_landed_leg_per,
#             "r_landed_dist_per" : r_landed_dist_per,
#             "r_landed_clinch_per" : r_landed_clinch_per,
#             "r_landed_ground_per" : r_landed_ground_per,
#             "b_kd" : b_kd,
#             "b_sig_str_landed" : b_sig_str_landed,
#             "b_sig_str_atmpted" : b_sig_str_atmpted,
#             "b_sig_str_acc" : b_sig_str_acc,
#             "b_total_str_landed" : b_total_str_landed,
#             "b_total_str_atmpted" : b_total_str_atmpted,
#             "b_total_str_acc" : b_total_str_acc,
#             "b_td_landed" : b_td_landed,
#             "b_td_atmpted" : b_td_atmpted,
#             "b_td_acc" : b_td_acc,
#             "b_sub_att" : b_sub_att,
#             "b_ctrl" : b_ctrl,
#             "b_head_landed" : b_head_landed,
#             "b_head_atmpted" : b_head_atmpted,
#             "b_head_acc" : b_head_acc,
#             "b_body_landed" : b_body_landed,
#             "b_body_atmpted" : b_body_atmpted,
#             "b_body_acc" : b_body_acc,
#             "b_leg_landed" : b_leg_landed,
#             "b_leg_atmpted" : b_leg_atmpted,
#             "b_leg_acc" : b_leg_acc,
#             "b_dist_landed" : b_dist_landed,
#             "b_dist_atmpted" : b_dist_atmpted,
#             "b_dist_acc" : b_dist_acc,
#             "b_clinch_landed" : b_clinch_landed,
#             "b_clinch_atmpted" : b_clinch_atmpted,
#             "b_clinch_acc" : b_clinch_acc,
#             "b_ground_landed" : b_ground_landed,
#             "b_ground_atmpted" : b_ground_atmpted,
#             "b_ground_acc" : b_ground_acc,
#             "b_landed_head_per" : b_landed_head_per,
#             "b_landed_body_per" : b_landed_body_per,
#             "b_landed_leg_per" : b_landed_leg_per,
#             "b_landed_dist_per" : b_landed_dist_per,
#             "b_landed_clinch_per" : b_landed_clinch_per,
#             "b_landed_ground_per" : b_landed_ground_per
#         }
#         with lock:
#             fight_details.append(data_dic)
#             # print(f"Scraped {idx+1}/{len(new_fight_links_all)}: {link}")
#             idx += 1
#     except Exception as e:
#         print(f"FAILED [{idx}] {link}")
#         print(f"{type(e).__name__}: {e}")
#         return

# with ThreadPoolExecutor(max_workers= MAX_THREADS) as executor:
#     results = [executor.submit(get_fight_data, item) for item in enumerate(new_fight_links_all)]
#     for r in results:
#         r.result()
        
# print(f"Successfully scraped all fight data. Scrapped data {len(fight_details)}")

In [12]:
# df_fight = pd.DataFrame(data=fight_details)
# df_fight.to_csv("fight_details.csv", index = False)
# df_fight

In [22]:
process = subprocess.Popen(
    [sys.executable, "scrape_fights.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1
)

for line in process.stdout:
    print(line, end="")

process.wait()

print("Finished with code:", process.returncode)

if process.returncode != 0:
    raise RuntimeError("scrape_fights.py failed")

df_fight = pd.read_csv(backend_dir / "data" / "fight_details.csv")
print(f"Fight records loaded: {len(df_fight)}")

Loaded 8832 fight links
Found 8675 existing fights — will skip these and append new ones
159 fights remaining to scrape

Scraping 159 fights with 6 concurrent browser workers...

Fights:   0%|          | 0/159 [00:00<?, ?it/s]FAILED [261] - Title: Loading… HTML: 2994FAILED [224] - Title: Loading… HTML: 2994


Fights:   1%|          | 1/159 [00:02<07:49,  2.97s/it]FAILED [47] - Title: Loading… HTML: 2994

Fights:   2%|▏         | 3/159 [00:03<02:09,  1.20it/s]FAILED [55] - Title: Loading… HTML: 2994
FAILED [31] - Title: Loading… HTML: 2994
FAILED [619] http://ufcstats.com/fight-details/1158a27a1a04b971 — Playwright could not load page: Error
FAILED [621] http://ufcstats.com/fight-details/1162a74e3604c488 — Playwright could not load page: Error
FAILED [622] http://ufcstats.com/fight-details/1168a8e59aacfd72 — Playwright could not load page: Error
FAILED [626] http://ufcstats.com/fight-details/118c25bda9392140 — Playwright could not load page: Error
FAILED [448] - Title: Loading… HTML: 29

# Scraping the fighter info

In [24]:
process = subprocess.Popen(
    [sys.executable, "scrape_fighter_details.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
    bufsize=1
)

for line in process.stdout:
    print(line, end="")

process.wait()

print("Finished with code:", process.returncode)

if process.returncode != 0:
    raise RuntimeError("scrape_fighter_details.py failed")

df_fighter = pd.read_csv(backend_dir / "data" / "fighter_details.csv")
print(f"Fighter records loaded: {len(df_fighter)}")

fighter_details.csv is empty — starting fresh
2724 unique fighter IDs found in fight data
Found 258 fighters flagged for refresh (had a new fight this cycle)
Never scraped before: 2724
Needing refresh from new fights: 258
2724 fighters remaining to scrape

Scraping 2724 fighters with 6 concurrent browser workers...

Fighters:   0%|          | 0/2724 [00:00<?, ?it/s]FAILED http://ufcstats.com/fighter-details/008ea710276c9606 — page did not contain expected fighter title (possibly blocked or not loaded)

Fighters:   0%|          | 1/2724 [00:01<1:03:51,  1.41s/it]FAILED http://ufcstats.com/fighter-details/003d82fa384ca1d0 — page did not contain expected fighter title (possibly blocked or not loaded)
FAILED http://ufcstats.com/fighter-details/0052de90691d4a93 — page did not contain expected fighter title (possibly blocked or not loaded)

Fighters:   0%|          | 2/2724 [00:01<41:02,  1.11it/s]  FAILED http://ufcstats.com/fighter-details/001eb2ab0f30e7ea — page did not contain expected f

In [25]:
df_fighter

,id,name,nick_name,wins,losses,draws,height,weight,reach,stance,dob,splm,str_acc,sapm,str_def,td_avg,td_avg_acc,td_def,sub_avg
0,0112352cb32f5026,Denis Tiuliulin,NaN,10,10,0,185.42,83.91,195.58,Orthodox,"May 17, 1988",3.61,41,5.23,38,0.96,42,72,0.0
1,013da757877044a2,Joe Brammer,The South Side Strangler,7,3,1,172.72,70.31,NaN,Orthodox,"Aug 23, 1983",2.15,31,2.80,56,0.81,50,50,0.0
2,00e11b5c8b7bfeeb,Luke Rockhold,NaN,16,6,0,190.50,83.91,195.58,Southpaw,"Oct 17, 1984",4.10,49,2.68,53,0.70,29,65,1.0
3,01641ba5df0c69b0,Gabriel Bonfim,Marretinha,20,1,0,185.42,77.11,182.88,Orthodox,"Aug 20, 1997",4.76,47,3.61,64,2.57,53,78,1.0
4,01d2ed8c502e3828,Caros Fodor,The Future,11,6,0,175.26,70.31,187.96,Orthodox,"Jan 07, 1984",2.76,54,2.83,54,2.10,25,50,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2677,ffc3e6daaa6da0b7,Johnny Bedford,Brutal,21,13,1,177.80,61.23,180.34,Orthodox,"Jan 06, 1983",4.53,50,1.75,63,1.83,62,62,0.0
2678,ffe9703408fb5964,David Onama,NaN,14,3,0,180.34,65.77,187.96,Orthodox,"Jun 07, 1994",5.07,50,4.89,51,1.05,30,52,0.5
2679,ffd3224638c01b57,Jean Matsumoto,NaN,18,2,0,167.64,61.23,172.72,Orthodox,"Sep 09, 1999",5.35,38,5.29,49,2.85,42,56,0.5
2680,ffdeb4fbea09ce75,Jessica Rakoczy,The Ragin',1,5,0,170.18,52.16,NaN,Orthodox,"Apr 14, 1977",2.50,44,3.55,64,0.75,20,0,0.8


# Building the final data, by merging the tables 

In [ ]:
# ============================================================
# RESTORE UFC.CSV TO THE EXACT OLD SCHEMA
# ============================================================

# ------------------------------------------------------------
# 1. Restore date/location from the duplicate merge columns
# ------------------------------------------------------------

# If date/location already exist, keep them.
# Otherwise combine the duplicated versions.
if 'date' not in df_fight.columns:
    if 'date_x' in df_fight.columns and 'date_y' in df_fight.columns:
        df_fight['date'] = df_fight['date_x'].combine_first(df_fight['date_y'])
    elif 'date_x' in df_fight.columns:
        df_fight['date'] = df_fight['date_x']
    elif 'date_y' in df_fight.columns:
        df_fight['date'] = df_fight['date_y']

if 'location' not in df_fight.columns:
    if 'location_x' in df_fight.columns and 'location_y' in df_fight.columns:
        df_fight['location'] = df_fight['location_x'].combine_first(df_fight['location_y'])
    elif 'location_x' in df_fight.columns:
        df_fight['location'] = df_fight['location_x']
    elif 'location_y' in df_fight.columns:
        df_fight['location'] = df_fight['location_y']


# ------------------------------------------------------------
# 2. Remove the duplicate merge columns
# ------------------------------------------------------------

df_fight = df_fight.drop(
    columns=[
        'date_x',
        'date_y',
        'location_x',
        'location_y'
    ],
    errors='ignore'
)


# ------------------------------------------------------------
# 3. Identify fighter columns
# ------------------------------------------------------------

r_cols = [c for c in df_fight.columns if c.startswith('r_')]
b_cols = [c for c in df_fight.columns if c.startswith('b_')]


# ------------------------------------------------------------
# 4. Recreate the old UFC.csv column order
# ------------------------------------------------------------

base_cols = [
    'event_id',
    'event_name',
    'date',
    'location',
    'fight_id',
    'division',
    'title_fight',
    'method',
    'finish_round',
    'match_time_sec',
    'total_rounds',
    'referee'
]

# Everything after referee was:
# red fighter columns -> blue fighter columns -> winner data

old_order = (
    base_cols
    + r_cols
    + b_cols
    + ['winner', 'winner_id']
)


# ------------------------------------------------------------
# 5. Verify nothing is missing BEFORE reordering
# ------------------------------------------------------------

missing = [c for c in old_order if c not in df_fight.columns]

if missing:
    print("ERROR: These columns are missing:")
    print(missing)
    raise KeyError(f"Missing columns: {missing}")


# ------------------------------------------------------------
# 6. Reorder
# ------------------------------------------------------------

df_fight = df_fight[old_order]


# ------------------------------------------------------------
# 7. Convert date/DOB exactly as before
# ------------------------------------------------------------

df_fight['date'] = pd.to_datetime(
    df_fight['date'],
    errors='coerce'
)

if 'r_dob' in df_fight.columns:
    df_fight['r_dob'] = pd.to_datetime(
        df_fight['r_dob'],
        errors='coerce'
    )

if 'b_dob' in df_fight.columns:
    df_fight['b_dob'] = pd.to_datetime(
        df_fight['b_dob'],
        errors='coerce'
    )


# ------------------------------------------------------------
# 8. Save
# ------------------------------------------------------------

output_path = backend_dir / "data" / "UFC.csv"

df_fight.to_csv(
    output_path,
    index=False
)


# ------------------------------------------------------------
# 9. Verify
# ------------------------------------------------------------

print("=" * 70)
print("UFC.CSV CREATED")
print("=" * 70)
print(f"Rows:    {len(df_fight)}")
print(f"Columns: {len(df_fight.columns)}")
print(f"Saved:   {output_path}")

print("\nFirst 20 columns:")
print(df_fight.columns[:20].tolist())

print("\nLast 5 columns:")
print(df_fight.columns[-5:].tolist())

print("\nDuplicate date/location columns:")
print([
    c for c in df_fight.columns
    if c in ['date_x', 'date_y', 'location_x', 'location_y']
])

df_fight

KeyError: "['date', 'location'] not in index"